In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

session.sql("USE ROLE POC_DEVELOPER").collect()

session.sql("USE DATABASE POC_POLICY").collect()
session.sql("USE SCHEMA PROCUREMENT_POLICY").collect()


In [ ]:
import requests

def test_connectivity():
    test_url = "https://www.info.buy.nsw.gov.au"
    try:
        response = requests.get(test_url, timeout=10)
        print(f"Connection successful!")
        print(f"   Status code : {response.status_code}")
        print(f"   Response size: {len(response.content)} bytes")
        return True
    except requests.exceptions.ConnectionError as e:
        print(f"Connection failed - Network rule may not be attached to notebook")
        print(f"   Error: {e}")
        return False
    except requests.exceptions.Timeout:
        print(f"Connection timed out")
        return False
    except Exception as e:
        print(f"Unexpected error: {e}")
        return False

test_connectivity()

In [ ]:
-- CREATE OR REPLACE TABLE BUY_NSW_CATALOG (
--     SOURCE         VARCHAR,        -- 'Policy' or 'Scheme'
--     NAME           VARCHAR,        -- policy/scheme name
--     UPDATED_DATE   TIMESTAMP_NTZ,
--     SCHEME_NUMBER  VARCHAR,        -- e.g. SCM2701 (N/A for policies)
--     TYPE           VARCHAR,        -- policy type  (N/A for schemes)
--     STATUS         VARCHAR,        -- e.g. Active / Recommended
--     DESCRIPTION    VARCHAR,
--     URL            VARCHAR,
--     SCRAPED_AT     TIMESTAMP_NTZ,
--     LAST_MODIFIED_AT TIMESTAMP_NTZ
-- );

In [ ]:
-- -- Create email notification integration
-- CREATE OR REPLACE NOTIFICATION INTEGRATION BUYNSW_POLICY_EMAIL_INTEGRATION
--     TYPE = EMAIL
--     ENABLED = TRUE
--     ALLOWED_RECIPIENTS = ('shruti.sharma@billigence.com');


-- GRANT USAGE ON INTEGRATION BUYNSW_POLICY_EMAIL_INTEGRATION TO ROLE POC_DEVELOPER;

In [ ]:
CREATE OR REPLACE PROCEDURE CHECK_BUY_NSW_CATALOG()
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python', 'requests', 'beautifulsoup4', 'pandas', 'python-dateutil')
EXTERNAL_ACCESS_INTEGRATIONS = (BUYNSW_POLICY_INTEGRATION)
HANDLER = 'run'
AS
$$
import requests
from bs4 import BeautifulSoup
import pandas as pd
from snowflake.snowpark.context import get_active_session
from datetime import datetime
from dateutil import parser as dateparser


HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Referer":         "https://www.info.buy.nsw.gov.au/"
}


# ----------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------
def has_next_page(soup):
    pagination   = soup.select("div.pagination ul li")
    found_current = False
    for li in pagination:
        if "current" in li.get("class", []):
            found_current = True
        elif found_current:
            return True
    return False


def parse_date(raw_date):
    try:
        return dateparser.parse(raw_date) if raw_date else None
    except Exception:
        return None


def format_date(val):
    """Return a clean readable string from any date value.
    Safely handles None, pandas NaT, float nan, the sentinel string 'NEW',
    and real datetimes."""
    if val is None:
        return "Not available"
    if val == "NEW":
        return "New record"
    try:
        if pd.isnull(val):          # catches NaT and float nan
            return "Not available"
    except Exception:
        pass
    try:
        return pd.Timestamp(val).strftime("%d %b %Y")
    except Exception:
        return str(val)


# ----------------------------------------------------------------
# Scrape Policies
# ----------------------------------------------------------------
def scrape_nsw_policies():
    base_url = "https://www.info.buy.nsw.gov.au/policy-library/policy-library-search"
    params = {
        "action":     "557003",
        "clive":      "dcs2~ds-buy-nsw-library",
        "collection": "dcs2~sp-buy-nsw-search",
        "cookie":     "false",
        "form":       "wrapper",
        "num_ranks":  "100",
        "profile":    "_default",
        "query":      "",
        "show":       "true",
        "sort":       "date",
        "start_rank": "1"
    }

    all_results = []
    start_rank  = 1

    while True:
        params["start_rank"] = start_rank
        response = requests.get(base_url, params=params, headers=HEADERS, timeout=30)
        if response.status_code != 200:
            break

        soup    = BeautifulSoup(response.text, "html.parser")
        results = soup.select("div.list-block__item")
        if not results:
            break

        for item in results:
            name_tag     = item.select_one("div.list-block__item-title a")
            name         = name_tag.get_text(strip=True) if name_tag else "N/A"
            url          = name_tag["href"] if name_tag and name_tag.has_attr("href") else "N/A"

            date_tag     = item.select_one("div.list-block__item-date")
            raw_date     = date_tag.get_text(strip=True).replace("Updated:", "").strip() if date_tag else None
            updated_date = parse_date(raw_date)

            cats         = [c.get_text(strip=True) for c in item.select("ul.list-block__item-category li")]
            policy_type  = cats[0] if len(cats) > 0 else "N/A"
            status       = cats[1] if len(cats) > 1 else "N/A"

            desc_tag     = item.select_one("div.list-block__item-content p")
            description  = desc_tag.get_text(strip=True) if desc_tag else "N/A"

            all_results.append({
                "SOURCE":           "Policy",
                "NAME":             name,
                "UPDATED_DATE":     updated_date,
                "SCHEME_NUMBER":    "N/A",
                "TYPE":             policy_type,
                "STATUS":           status,
                "DESCRIPTION":      description,
                "URL":              url,
                "SCRAPED_AT":       datetime.now(),
                "LAST_MODIFIED_AT": datetime.now()   # equals SCRAPED_AT on first load
            })

        if has_next_page(soup):
            start_rank += 100
        else:
            break

    return all_results


# ----------------------------------------------------------------
# Scrape Schemes
# ----------------------------------------------------------------
def scrape_nsw_schemes():
    base_url = "https://www.info.buy.nsw.gov.au/schemes/schemes-search"
    params = {
        "action":     "570183",
        "clive":      "dcs2~ds-buy-nsw-schemes",
        "collection": "dcs2~sp-buy-nsw-search",
        "cookie":     "false",
        "form":       "wrapper",
        "num_ranks":  "100",
        "profile":    "_default",
        "query":      "",
        "show":       "true",
        "sort":       "metat",
        "start_rank": "1"
    }

    all_results = []
    start_rank  = 1

    while True:
        params["start_rank"] = start_rank
        response = requests.get(base_url, params=params, headers=HEADERS, timeout=30)
        if response.status_code != 200:
            break

        soup    = BeautifulSoup(response.text, "html.parser")
        results = soup.select("div.list-block__item")
        if not results:
            break

        for item in results:
            name_tag      = item.select_one("div.list-block__item-title a")
            name          = name_tag.get_text(strip=True) if name_tag else "N/A"
            url           = name_tag["href"] if name_tag and name_tag.has_attr("href") else "N/A"

            date_tag      = item.select_one("div.list-block__item-date")
            raw_date      = date_tag.get_text(strip=True).replace("Updated:", "").strip() if date_tag else None
            updated_date  = parse_date(raw_date)

            cats          = [c.get_text(strip=True) for c in item.select("ul.list-block__item-category li")]
            scheme_number = cats[0] if len(cats) > 0 else "N/A"
            status        = cats[1] if len(cats) > 1 else "N/A"

            desc_tag      = item.select_one("div.list-block__item-content p")
            description   = desc_tag.get_text(strip=True) if desc_tag else "N/A"

            all_results.append({
                "SOURCE":           "Scheme",
                "NAME":             name,
                "UPDATED_DATE":     updated_date,
                "SCHEME_NUMBER":    scheme_number,
                "TYPE":             "N/A",
                "STATUS":           status,
                "DESCRIPTION":      description,
                "URL":              url,
                "SCRAPED_AT":       datetime.now(),
                "LAST_MODIFIED_AT": datetime.now()   # equals SCRAPED_AT on first load
            })

        if has_next_page(soup):
            start_rank += 100
        else:
            break

    return all_results


# ----------------------------------------------------------------
# Compare scraped data against BUY_NSW_CATALOG
# ----------------------------------------------------------------
def check_for_updates(session, latest_df):
    try:
        existing_df = session.table("BUY_NSW_CATALOG").to_pandas()
    except Exception:
        existing_df = pd.DataFrame()

    if existing_df.empty:
        sdf = session.create_dataframe(latest_df)
        sdf.write.mode("truncate").save_as_table("BUY_NSW_CATALOG")
        return []

    # Normalise column names from SF (may come back uppercase)
    existing_df.columns = [c.upper() for c in existing_df.columns]
    latest_df.columns   = [c.upper() for c in latest_df.columns]

    # Drop records where the website has no date — can't meaningfully compare them
    latest_df = latest_df[latest_df["UPDATED_DATE"].notna()].reset_index(drop=True)

    merged = latest_df.merge(
        existing_df[["SOURCE", "NAME", "UPDATED_DATE", "LAST_MODIFIED_AT"]],
        on=["SOURCE", "NAME"],
        suffixes=("_WEB", "_SF")
    )

    changed = merged[merged["UPDATED_DATE_WEB"] != merged["UPDATED_DATE_SF"]].copy()
    changed["OLD_DATE"] = changed["UPDATED_DATE_SF"]
    changed["NEW_DATE"] = changed["UPDATED_DATE_WEB"]

    existing_keys = set(zip(existing_df["SOURCE"], existing_df["NAME"]))
    latest_keys   = list(zip(latest_df["SOURCE"],  latest_df["NAME"]))
    new_mask      = [k not in existing_keys for k in latest_keys]
    new_records   = latest_df[new_mask].copy()
    new_records["OLD_DATE"] = "NEW"
    new_records["NEW_DATE"] = new_records["UPDATED_DATE"]

    all_changes = pd.concat([changed, new_records], ignore_index=True)

    # ---------------------------------------------------------------
    # Build the final dataframe to write back to Snowflake:
    #   - unchanged records  → keep their existing LAST_MODIFIED_AT
    #   - changed / new ones → stamp datetime.now()
    # ---------------------------------------------------------------
    changed_keys = set(zip(all_changes["SOURCE"], all_changes["NAME"])) if not all_changes.empty else set()

    def pick_last_modified(row):
        key = (row["SOURCE"], row["NAME"])
        if key in changed_keys:
            return datetime.now()   # code just updated UPDATED_DATE → stamp now
        # unchanged — carry over what was in SF (fall back to SCRAPED_AT if missing)
        match = existing_df[
            (existing_df["SOURCE"] == row["SOURCE"]) &
            (existing_df["NAME"]   == row["NAME"])
        ]
        if not match.empty and pd.notna(match.iloc[0]["LAST_MODIFIED_AT"]):
            return match.iloc[0]["LAST_MODIFIED_AT"]
        return row["SCRAPED_AT"]

    latest_df["LAST_MODIFIED_AT"] = latest_df.apply(pick_last_modified, axis=1)

    if not all_changes.empty:
        sdf = session.create_dataframe(latest_df)
        sdf.write.mode("truncate").save_as_table("BUY_NSW_CATALOG")

    return all_changes.to_dict("records")


# ----------------------------------------------------------------
# Send email notification
# ----------------------------------------------------------------
def send_notification(session, changes):
    if not changes:
        return

    policy_changes = [c for c in changes if c.get("SOURCE") == "Policy"]
    scheme_changes = [c for c in changes if c.get("SOURCE") == "Scheme"]

    def build_rows(change_list):
        rows = ""
        for c in change_list:
            rows += (
                "<tr>"
                f"<td style='padding:8px;border:1px solid #ddd'>{c.get('NAME','N/A')}</td>"
                f"<td style='padding:8px;border:1px solid #ddd'>{c.get('SCHEME_NUMBER','N/A')}</td>"
                f"<td style='padding:8px;border:1px solid #ddd'>{format_date(c.get('OLD_DATE'))}</td>"
                f"<td style='padding:8px;border:1px solid #ddd'>{format_date(c.get('NEW_DATE'))}</td>"
                f"<td style='padding:8px;border:1px solid #ddd'><a href='{c.get('URL','#')}'>View</a></td>"
                "</tr>"
            )
        return rows

    def build_section(title, change_list):
        if not change_list:
            return ""
        return (
            f"<h3 style='color:#333'>{title}</h3>"
            "<table style='border-collapse:collapse;width:100%;margin-bottom:24px'>"
            "<tr style='background:#f2f2f2'>"
            "<th style='padding:8px;border:1px solid #ddd'>Name</th>"
            "<th style='padding:8px;border:1px solid #ddd'>Scheme No.</th>"
            "<th style='padding:8px;border:1px solid #ddd'>Previous Date</th>"
            "<th style='padding:8px;border:1px solid #ddd'>New Date</th>"
            "<th style='padding:8px;border:1px solid #ddd'>Link</th>"
            "</tr>"
            f"{build_rows(change_list)}"
            "</table>"
        )

    run_time   = datetime.now().strftime('%d %b %Y %H:%M')
    total      = len(changes)
    html_body  = (
        "<html><body style='font-family:Arial,sans-serif;color:#333'>"
        "<h2>BUY NSW Catalog Updates Detected</h2>"
        f"<p>The following policies and/or schemes have been updated or added as of <b>{run_time}</b>:</p>"
        f"{build_section('Policy Changes', policy_changes)}"
        f"{build_section('Scheme Changes', scheme_changes)}"
        "<p>Please review and update your records accordingly.</p>"
        "<p><i>This is an automated notification from Snowflake.</i></p>"
        "</body></html>"
    )

    # Escape single quotes for the SQL string literal
    safe_body  = html_body.replace("'", "''")
    subject    = f"BUY NSW Catalog Updates Detected — {total} change(s)"

    session.sql(f"""
        CALL SYSTEM$SEND_EMAIL(
            'BUYNSW_POLICY_EMAIL_INTEGRATION',
            'shruti.sharma@billigence.com',
            '{subject}',
            '{safe_body}',
            'text/html'
        )
    """).collect()


# ----------------------------------------------------------------
# HANDLER  — Snowflake calls run(session) automatically
# ----------------------------------------------------------------
def run(session):
    try:
        policy_records = scrape_nsw_policies()
        scheme_records = scrape_nsw_schemes()

        all_records = policy_records + scheme_records
        if not all_records:
            return "ERROR: No records scraped from either source."

        latest_df = pd.DataFrame(all_records)
        changes   = check_for_updates(session, latest_df)
        send_notification(session, changes)

        policy_count = len(policy_records)
        scheme_count = len(scheme_records)
        change_count = len(changes)

        return (
            f"OK | Policies: {policy_count} | Schemes: {scheme_count} | "
            f"Changes: {change_count} | Run: {datetime.now().strftime('%d %b %Y %H:%M')}"
        )

    except Exception as e:
        return f"ERROR: {str(e)}"
$$;



In [ ]:
CALL CHECK_BUY_NSW_CATALOG();

In [ ]:
--Runs the procedure daily at 5 AM Sydney time

CREATE OR REPLACE TASK CHECK_BUY_NSW_CATALOG_TASK
    WAREHOUSE = POC           
    SCHEDULE  = 'USING CRON 0 19 * * * UTC'   
    COMMENT   = 'Daily scrape of BUY NSW Policy and Scheme Library at 5 AM Sydney time'
AS
    CALL CHECK_BUY_NSW_CATALOG();


ALTER TASK CHECK_BUY_NSW_CATALOG_TASK RESUME;

EXECUTE TASK CHECK_BUY_NSW_CATALOG_TASK RESUME;

